In [1]:
# Standard library imports
import os
import random
import sys
from itertools import product

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure matplotlib
plt.style.use('../fig.style')
figsize = (8,4)

# Add project root to path
project_root = os.path.abspath('../src/')
if project_root not in sys.path:
    sys.path.append(project_root)
    
# Project imports
from gaugefixer.features import get_allorder_features
from gaugefixer.fixers import fix_allorder_model
from gaugefixer.verify import verify_identical_evaluation, verify_marginalization
from gaugefixer.sequence import SeqEmbedder, get_alphabet, evaluate_model_on_seqs, randseq, randseqs

evaluate_on_seqs = False

In [2]:
L = 9
alphabet = get_alphabet('dna')
alpha = len(alphabet)
pi_lc = np.ones((L,alpha))/alpha

# Create all-order model features and corresponding embedder
features = get_allorder_features(L=L, alphabet=alphabet)
N = len(features)
print(f'{N=:,}')

# Create embedder
if evaluate_on_seqs:
    embedder = SeqEmbedder(features=features, L=L)

# Create theta_series
theta_series = pd.Series(index=features, data=np.random.normal(size=N))
theta_series.tail()

N=1,953,125


((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTGT)   -0.600296
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTA)    0.006227
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTC)    1.172764
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTG)    1.465545
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTT)   -0.587727
dtype: float64

In [3]:
fixed_theta_series = fix_allorder_model(theta=theta_series, 
                                        L=L, 
                                        alphabet=alphabet,
                                        gauge='zero-sum')
fixed_theta_series.tail()

((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTGT)   -0.207450
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTA)   -0.497624
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTC)    0.571998
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTG)    0.429823
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTT)   -0.504197
dtype: float64

In [7]:
fixed_theta_series = fix_allorder_model(theta=theta_series, 
                                        L=L, 
                                        alphabet=alphabet,
                                        gauge='wild-type',
                                        wt_seq=randseq(L=L, alphabet=alphabet))
fixed_theta_series.tail()

((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTGT)   -36.683505
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTA)     0.000000
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTC)     0.000000
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTG)     0.000000
((0, 1, 2, 3, 4, 5, 6, 7, 8), TTTTTTTTT)     0.000000
dtype: float64

In [5]:
if evaluate_on_seqs:
    random_seqs = randseqs(num_seqs=100, L=L, alphabet=alphabet)
    evaluate_model_on_seqs(theta=theta_series,
                        seqs=random_seqs,
                        embedder=embedder)

In [6]:
kwargs = dict(
    wt_seq = randseq(L, alphabet),
    gauge = 'wild-type',
    pi_lc = None,
    L=L,
    alphabet=alphabet
)

fixed_theta_series = fix_allorder_model(theta=theta_series, **kwargs)


verify_marginalization(
    theta_series=fixed_theta_series,
    num_tests=100,
    **kwargs)
        
if evaluate_on_seqs:
    verify_identical_evaluation(
        theta1=theta_series, 
        theta2=fixed_theta_series, 
        L=L, 
        alphabet=alphabet, 
        num_tests=100)

test_num=0: order=4, orbit=(1, 6, 7, 8), pos=0, seq_to_show='*TTA', Passed
test_num=1: order=2, orbit=(2, 8), pos=0, seq_to_show='*G', Passed
test_num=2: order=8, orbit=(0, 1, 2, 3, 5, 6, 7, 8), pos=2, seq_to_show='AA*TACCG', Passed
test_num=3: order=5, orbit=(0, 1, 2, 3, 7), pos=2, seq_to_show='GT*TC', Passed
test_num=4: order=9, orbit=(0, 1, 2, 3, 4, 5, 6, 7, 8), pos=4, seq_to_show='CATG*GGAG', Passed
test_num=5: order=3, orbit=(3, 5, 6), pos=0, seq_to_show='*GT', Passed
test_num=6: order=4, orbit=(2, 4, 6, 8), pos=1, seq_to_show='G*CC', Passed
test_num=7: order=3, orbit=(1, 6, 8), pos=1, seq_to_show='G*A', Passed
test_num=8: order=1, orbit=(0,), pos=0, seq_to_show='*', Passed
test_num=9: order=7, orbit=(0, 1, 2, 3, 4, 5, 7), pos=2, seq_to_show='GT*AGGA', Passed
test_num=10: order=3, orbit=(1, 5, 8), pos=2, seq_to_show='GG*', Passed
test_num=11: order=3, orbit=(1, 2, 7), pos=0, seq_to_show='*GG', Passed
test_num=12: order=6, orbit=(0, 1, 3, 4, 7, 8), pos=2, seq_to_show='AC*AAT', Pass